In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURATION: Change everything from here
# ==========================================
CONFIG = {
    'seed': 42,
    # We will detect the target_col automatically below, so you can leave this blank
    'target_col': None,
    'n_splits': 5,
    'train_path': '/kaggle/input/competitions/mle-practice-test/train.csv',
    'test_path': '/kaggle/input/competitions/mle-practice-test/test.csv',
    'sub_name': 'submission_regression.csv'
}

# ==========================================
# PHASE 1: DATA LOADING & TARGET DETECTION
# ==========================================
print("--> Loading data...")
train = pd.read_csv(CONFIG['train_path'])
test = pd.read_csv(CONFIG['test_path'])

# INSTRUCTIONS TO FIND THE TARGET COLUMN:
# 1. The target column is what the model needs to predict.
# 2. It will always be present in your training data, but missing from your test data.
# 3. We can find it programmatically by checking the difference between train and test columns.

# Find the difference between train columns and test columns
detected_targets = list(set(train.columns) - set(test.columns))

# Set the target column automatically
CONFIG['target_col'] = detected_targets[0]
print(f"--> Auto-detected target column: '{CONFIG['target_col']}'")

# Split features and target
X = train.drop(columns=[CONFIG['target_col']])
y = train[CONFIG['target_col']]
X_test = test.copy()

# ==========================================
# PHASE 2: ADVANCED FEATURE ENGINEERING
# ==========================================
def engineer_features(df):
    df = df.copy()
    numeric_cols = df.select_dtypes(include=np.number).columns

    # Row-wise statistics
    df['row_mean'] = df[numeric_cols].mean(axis=1)
    df['row_std'] = df[numeric_cols].std(axis=1)

    return df, numeric_cols

print("--> Engineering features...")
X, num_cols = engineer_features(X)
X_test, _ = engineer_features(X_test)

# KMeans Clustering (fit on entire X to learn global structure)
kmeans = KMeans(n_clusters=5, random_state=CONFIG['seed'], n_init=10)
X['patient_cluster'] = kmeans.fit_predict(X[num_cols].fillna(0))
X_test['patient_cluster'] = kmeans.predict(X_test[num_cols].fillna(0))

# ==========================================
# PHASE 3: PREPROCESSING PIPELINE
# ==========================================
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

if 'patient_cluster' not in categorical_features:
    categorical_features.append('patient_cluster')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_features)
    ])

# ==========================================
# PHASE 4: MODEL DEFINITION
# ==========================================
# Using aggressive hyperparameters for regression
lgbm = LGBMRegressor(n_estimators=500, learning_rate=0.02, num_leaves=31, random_state=CONFIG['seed'], verbose=-1)
xgb = XGBRegressor(n_estimators=500, learning_rate=0.02, max_depth=6, random_state=CONFIG['seed'], eval_metric='rmse', verbosity=0)
rf = RandomForestRegressor(n_estimators=500, max_depth=12, random_state=CONFIG['seed'])

# VotingRegressor automatically averages the predictions of all 3 models
ensemble = VotingRegressor(
    estimators=[('lgbm', lgbm), ('xgb', xgb), ('rf', rf)]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', ensemble)
])

# ==========================================
# PHASE 5: CROSS-VALIDATION (DEBUG & EVAL)
# ==========================================
print("\n--> Starting 5-Fold Cross Validation...")
kf = KFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=CONFIG['seed'])

fold_rmses = []
fold_r2s = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

    # Train on 80% fold
    pipeline.fit(X_tr, y_tr)

    # Predict on 20% fold
    preds = pipeline.predict(X_va)
    rmse = np.sqrt(mean_squared_error(y_va, preds))
    r2 = r2_score(y_va, preds)

    fold_rmses.append(rmse)
    fold_r2s.append(r2)
    print(f"Fold {fold} | RMSE: {rmse:.4f} | R2 Score: {r2:.4f}")

print(f"---")
print(f"Mean CV RMSE: {np.mean(fold_rmses):.4f}")
print(f"Mean CV R2:   {np.mean(fold_r2s):.4f}")
print(f"---\n")

# ==========================================
# PHASE 6: 100% DATA TRAINING & SUBMISSION
# ==========================================
print("--> Retraining ensemble on 100% of the training data...")
pipeline.fit(X, y)
print("--> Training complete!")

print("--> Generating final predictions for test set...")
test_preds = pipeline.predict(X_test)

submission = pd.DataFrame({
    'id': test['id'],
    CONFIG['target_col']: test_preds
})

submission.to_csv(CONFIG['sub_name'], index=False)
print(f"--> Saved {CONFIG['sub_name']} with {len(submission)} rows!")